# Taylor 1.50: the skateboard beyond small angles

*HW01 starter notebook, PHY 317.* Run each cell in order with **Shift-Enter**. You only need to edit the cells marked **your turn**.

In Example 1.2 the skateboard in the half-pipe obeys
$$\ddot\phi = -\frac{g}{R}\sin\phi ,$$
and we got simple harmonic motion by replacing $\sin\phi$ with $\phi$. That gave
$$\phi(t) = \phi_0 \cos(\omega_0 t), \qquad \omega_0 = \sqrt{g/R}.$$
Here we'll use computational methods to solve the *real* equation, and you'll find out how large the release angle $\phi_0$ can be before the small-angle approximation breaks down meaningfully.

## 1. Predict first (before you run anything)

Double-click the cell below this one and type your answers, then Shift-Enter to set the text. Two or three sentences is plenty.

- At roughly what release angle $\phi_0$ do you expect the real motion and the small-angle formula to differ visibly after a few swings?
- Will the real skateboard take **longer** or **shorter** per swing than the small-angle formula predicts? Why?

*(your prediction here)*



In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact

g = 9.8    # m/s^2
R = 5.0    # m, radius of the half-pipe
t_max = 20.0   # seconds of motion to compute

## 2. The two solutions

`full_solution` feeds the real equation to a numerical solver. Different solvers work differently. This one wants the equation as two first-order equations, so the "state" is the pair (angle, angular velocity): the rate of change of the angle is the angular velocity, and the rate of change of the angular velocity is $-(g/R)\sin\phi$.

`small_angle_solution` is just the formula $\phi_0\cos(\omega_0 t)$.

In [ ]:
def full_solution(phi0_deg):
    """Solve the real equation. Released from rest at phi0 (degrees).
    Returns times t (seconds) and angles phi (degrees)."""
    phi0 = np.radians(phi0_deg)

    def rates(t, state):
        phi, phidot = state
        return [phidot, -(g / R) * np.sin(phi)]

    t = np.linspace(0, t_max, 4000)
    sol = solve_ivp(rates, (0, t_max), [phi0, 0.0], t_eval=t, rtol=1e-9)
    return t, np.degrees(sol.y[0])


def small_angle_solution(phi0_deg, t):
    omega0 = np.sqrt(g / R)
    return phi0_deg * np.cos(omega0 * t)

## 3. Move the slider

Solid = the real equation. Dashed = the small-angle approximation. Both start at the same angle; watch them drift apart.

In [ ]:
@interact(phi0_deg=(1, 179, 1))
def compare(phi0_deg=20):
    t, phi = full_solution(phi0_deg)
    plt.figure(figsize=(8, 3.5))
    plt.plot(t, phi, label="real equation")
    plt.plot(t, small_angle_solution(phi0_deg, t), "--", label="small angle")
    plt.xlabel("t (seconds)")
    plt.ylabel("phi (degrees)")
    plt.legend(loc="upper right")
    plt.grid(alpha=0.3)
    plt.show()

## 4. Period versus release angle

A cleaner way to see the breakdown: measure the actual period of the real solution (the time between two successive downward crossings of $\phi = 0$) and compare it with the small-angle period $T_0 = 2\pi/\omega_0$, which does not depend on $\phi_0$ at all.

In [ ]:
def measured_period(phi0_deg):
    t, phi = full_solution(phi0_deg)
    crossing = (phi[:-1] > 0) & (phi[1:] <= 0)     # True where phi goes from + to -
    times = t[:-1][crossing]
    return times[1] - times[0]

T0 = 2 * np.pi / np.sqrt(g / R)
print(f"small-angle period T0 = {T0:.3f} s")

# your turn: change the list of release angles
angles = [5, 10, 20, 30, 45, 60, 90, 120, 150]

periods = [measured_period(a) for a in angles]
plt.plot(angles, periods, "o-", label="real period")
plt.axhline(T0, color="k", linestyle="--", label="small-angle period T0")
plt.xlabel("release angle phi0 (degrees)")
plt.ylabel("period (seconds)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for a, T in zip(angles, periods):
    print(f"phi0 = {a:3d} deg:  period = {T:.3f} s   ({100 * (T - T0) / T0:+.1f}% vs T0)")

## 5. What you found

Double-click each answer cell, type, Shift-Enter.

**a. Compare with your prediction.** At what $\phi_0$ is the period off by more than 1%? Longer or shorter than the small-angle period, and why, from the sign of $\sin\phi - \phi$?

*(your answer here)*

**b. Assessment.** What is one check you can do to convince yourself that the numerical solution is correct (e.g. a limit where you already know what the answer should be).

*(your answer here)*

**c. What didn't work** the first time, if anything.

Then put your plots (screenshots are fine) and these answers in your homework PDF.

*(your answer here)*